# 전처리 전 과정 상세 (단계별 데이터)

probe→gene → 그룹라벨 → log2·정규화 → ComBat 을 **실제 데이터 전/후**로 확인.

In [1]:
import pandas as pd, gzip
from io import StringIO
pd.set_option('display.max_columns',12)
B='../01_preprocessing/bulk/output'; OLD='../../data/processed'
def read_series(g):
    with gzip.open(f'{OLD}/{g}_series_matrix.txt.gz','rt') as f: L=[next(f) for _ in range(120)]
    s=next(i for i,l in enumerate(L) if l.startswith('"ID_REF"'))
    return pd.read_csv(StringIO(''.join(L[s:s+11])),sep='\t',index_col=0)

## 전처리 1 — probe → gene (예: GSE30529)
마이크로어레이는 유전자 대신 **probe ID**로 나옴 (probe가 다양함).

In [2]:
raw=read_series('GSE30529')
print('원본: 행 = probe(다양한 ID), 열 = 샘플')
raw.iloc[:8,:4]

원본: 행 = probe(다양한 ID), 열 = 샘플


,GSM757014,GSM757015,GSM757016,GSM757017
ID_REF,,,,
1007_s_at,0.155938,0.095946,0.401841,0.516814
1053_at,0.236349,0.119733,0.081872,0.505905
117_at,-0.155251,0.173947,-0.033306,-0.058939
121_at,0.090568,0.440096,0.225213,-0.315482
1255_g_at,0.010099,-0.279639,-0.221435,0.097297
1294_at,-0.104819,0.650456,-0.014148,1.178083
1316_at,-0.247418,0.018172,-0.538636,-0.359284
1320_at,-0.068738,0.069029,-0.197250,0.477878


### 매핑 테이블 — 라이브러리(hgu133a2.db)가 probe→유전자 변환
우리는 R 라이브러리 `hgu133a2.db`가 매핑표를 **내장**. 예시(실제 표준 매핑):

In [3]:
mapping=pd.DataFrame({
 'probe':['1007_s_at','1053_at','117_at','121_at','1405_i_at','204655_at'],
 'gene' :['DDR1','RFC2','HSPA6','PAX8','CCL5','CCL5']})
print('※ 1405_i_at·204655_at 처럼 여러 probe가 같은 gene(CCL5)에 붙기도 함')
mapping

※ 1405_i_at·204655_at 처럼 여러 probe가 같은 gene(CCL5)에 붙기도 함


,probe,gene
0,1007_s_at,DDR1
1,1053_at,RFC2
2,117_at,HSPA6
3,121_at,PAX8
4,1405_i_at,CCL5
5,204655_at,CCL5


### 같은 gene은 평균 (avereps) → 유전자 1개 = 행 1개
probe 22,277개 → 유전자 13,041개 (같은 gene의 여러 probe를 평균해 합침)

In [4]:
res=pd.read_csv(f'{B}/01_genematrix/GSE30529.genematrix.txt',sep='\t',index_col=0)
n_probe=22277; n_gene=res.shape[0]
print(f'probe {n_probe} → 유전자 {n_gene}  (약 {n_probe-n_gene}개 probe가 평균으로 합쳐짐)')
res.iloc[:8,:4]

probe 22277 → 유전자 13041  (약 9236개 probe가 평균으로 합쳐짐)


,GSM757014,GSM757015,GSM757016,GSM757017
geneNames,,,,
DDR1,8.929265,9.193855,9.363028,9.078851
RFC2,5.655333,5.906418,5.581702,5.608812
HSPA6,5.702964,5.901149,5.408531,5.446837
PAX8,7.657273,6.882328,7.113481,7.172935
GUCA1A,4.389246,4.376325,4.395794,4.432834
UBA7,7.324307,7.844049,6.782658,6.589270
THRA,6.053758,5.872051,5.839461,5.823391
PTPN21,5.066210,5.393341,4.994323,5.113796


## 전처리 2 — 그룹 라벨 붙이기

각 샘플에 **Control / Early / Late** (또는 DKD/Control) 라벨을 열 이름에 붙임.

- **저자 방식**: `s1.txt`(Control)·`s2.txt`(Early)·`s3.txt`(Late) — 저자가 GEO 보고 만든 **샘플이름 목록**
- **우리 방식(현재)**: 로컬 `series_matrix` **메타데이터에서 자동 추출** (diabetic→DKD)  ※ 인터넷 불필요
  (이전엔 getGEO(인터넷) 썼는데 오프라인 실패 → series_matrix 메타로 바꿈)

In [5]:
# GSE30529: 라벨 전(GSM만) → 후(_DKD/_Control) — 전체 22개 샘플
before=pd.read_csv(f'{B}/01_genematrix/GSE30529.genematrix.txt',sep='\t',index_col=0)
after =pd.read_csv(f'{B}/02_normalized/GSE30529.normalized.txt',sep='\t',index_col=0)
from collections import Counter
print('series_matrix 메타로 판정:', dict(Counter(x.split('_')[-1] for x in after.columns)))
pd.DataFrame({'라벨 전':list(before.columns),'라벨 후':list(after.columns)})

series_matrix 메타로 판정: {'DKD': 10, 'Control': 12}


,라벨 전,라벨 후
0,GSM757014,GSM757014_DKD
1,GSM757015,GSM757015_DKD
2,GSM757016,GSM757016_DKD
3,GSM757017,GSM757017_DKD
4,GSM757018,GSM757018_DKD
5,GSM757019,GSM757019_DKD
6,GSM757020,GSM757020_DKD
7,GSM757021,GSM757021_DKD
8,GSM757022,GSM757022_DKD
9,GSM757023,GSM757023_DKD


In [6]:
# GSE142025: 접두사(A=Late/B=Early/N=Control)로 라벨 — 전체 36개 샘플
b=pd.read_csv(f'{B}/01_genematrix/GSE142025.genematrix.txt',sep='\t',index_col=0)
a=pd.read_csv(f'{B}/02_normalized/GSE142025.normalized.txt',sep='\t',index_col=0)
print('3그룹:', dict(Counter(x.split('_')[-1] for x in a.columns)))
pd.DataFrame({'라벨 전':list(b.columns),'라벨 후':list(a.columns)})

3그룹: {'Late': 21, 'Early': 6, 'Control': 9}


,라벨 전,라벨 후
0,A11A,A11A_Late
1,A12A,A12A_Late
2,A13A,A13A_Late
3,A14A,A14A_Late
4,A15A,A15A_Late
5,A17A,A17A_Late
6,A18A,A18A_Late
7,A19A,A19A_Late
8,A20A,A20A_Late
9,A21A,A21A_Late


## 전처리 2 (이어서) — log2 + 정규화

**정규화(normalizeBetweenArrays)** = 샘플마다 다른 **분포(밝기)** 를 동일하게 맞추는 것.

⚠️ 우리 데이터는 **받아왔을 때 이미 정규화된 상태**라 전/후가 같음:
- 어레이 → RMA가 이미 정규화 / RNA-seq(142025) → 제출자가 이미 정규화
- 그래서 아래처럼 **원리를 데모**로 보여줌 (일부러 샘플 밝기를 다르게 → 정규화로 복구)

In [7]:
# 정규화 원리 데모: 샘플 4개 중 2개를 일부러 밝기 다르게 → 정규화로 통일
g=pd.read_csv(f'{B}/01_genematrix/GSE142025.genematrix.txt',sep='\t',index_col=0)
demo=g.iloc[:,:4].copy()
demo.iloc[:,1]=demo.iloc[:,1]+2    # 샘플2 통째로 밝게(+2)
demo.iloc[:,2]=demo.iloc[:,2]*0.7  # 샘플3 어둡게(*0.7)
# quantile 정규화
rank_mean=demo.stack().groupby(demo.rank(method='first').stack().astype(int)).mean()
norm=demo.rank(method='min').stack().astype(int).map(rank_mean).unstack()
tbl=pd.DataFrame({'정규화 전 평균(뒤틀림)':demo.mean().round(3),
                  '정규화 후 평균(통일)':norm.mean().round(3)})
print('전: 샘플마다 평균 제각각(9.7/11.7/6.8/9.7) → 후: 다 같아짐(≈9.46)')
tbl

전: 샘플마다 평균 제각각(9.7/11.7/6.8/9.7) → 후: 다 같아짐(≈9.46)


,정규화 전 평균(뒤틀림),정규화 후 평균(통일)
A11A,9.694,9.464
A12A,11.694,9.465
A13A,6.786,9.463
A14A,9.694,9.464


In [8]:
# 우리 실제 142025: 이미 정규화됨 (샘플별 평균이 원래부터 다 동일)
n=pd.read_csv(f'{B}/02_normalized/GSE142025.normalized.txt',sep='\t',index_col=0)
proof=pd.DataFrame({'샘플평균':n.mean().round(3)})
print('전체 36샘플 평균이 다 ~9.694 = 이미 정규화됨 → 우리가 또 해도 안 변함')
proof

전체 36샘플 평균이 다 ~9.694 = 이미 정규화됨 → 우리가 또 해도 안 변함


,샘플평균
A11A_Late,9.694
A12A_Late,9.694
A13A_Late,9.694
A14A_Late,9.694
A15A_Late,9.694
A17A_Late,9.694
A18A_Late,9.694
A19A_Late,9.694
A20A_Late,9.694
A21A_Late,9.694


## 전처리 3 — ComBat 배치보정 (GSE104948 + GSE104954)
데이터셋 차이(배치효과) 제거. 보정 후 두 배치 값이 맞춰짐.

In [9]:
b1=pd.read_csv(f'{B}/02_normalized/GSE104948.normalized.txt',sep='\t',index_col=0)
b2=pd.read_csv(f'{B}/02_normalized/GSE104954.normalized.txt',sep='\t',index_col=0)
cb=pd.read_csv(f'{B}/03_merged/valid_104948_104954.combat.txt',sep='\t',index_col=0)
g1=set(x.split('_')[0] for x in b1.columns)
cb1=[c for c in cb.columns if c.split('_')[0] in g1]; cb2=[c for c in cb.columns if c.split('_')[0] not in g1]
gene='FN1'
tbl=pd.DataFrame({'배치1(104948) 평균':[round(b1.loc[gene].mean(),3),round(cb[cb1].loc[gene].mean(),3)],
                  '배치2(104954) 평균':[round(b2.loc[gene].mean(),3),round(cb[cb2].loc[gene].mean(),3)]},
                 index=['ComBat 전','ComBat 후'])
tbl['두 배치 차이']=abs(tbl['배치1(104948) 평균']-tbl['배치2(104954) 평균']).round(3)
print(f'{gene} 배치별 평균: 보정 후 차이 줄어듦(배치효과 제거)')
tbl

FN1 배치별 평균: 보정 후 차이 줄어듦(배치효과 제거)


,배치1(104948) 평균,배치2(104954) 평균,두 배치 차이
ComBat 전,7.434,5.946,1.488
ComBat 후,6.625,6.644,0.019


In [10]:
print('ComBat 후 최종:', cb.shape, '(12,042 유전자 × 71 샘플 = 33+38)')
cb.iloc[:5,:4]

ComBat 후 최종: (12042, 71) (12,042 유전자 × 71 샘플 = 33+38)


,GSM2810770_DKD,GSM2810771_DKD,GSM2810772_DKD,GSM2810773_DKD
geneNames,,,,
NAT2,7.058082,6.685972,5.777030,6.061398
ADA,6.260703,6.173888,6.079306,6.364291
CDH2,8.143298,8.443839,7.544486,8.113535
AKT3,6.321408,6.189759,6.731929,6.195660
MED6,6.774592,6.882633,7.023595,6.545485


## 정리 (전처리 4단계)
1. **probe→gene**: 라이브러리(hgu133a2.db)로 매핑, 같은 gene 평균(avereps)
2. **그룹라벨**: series_matrix 메타로 Control/Early/Late(또는 DKD/Control) — 인터넷 불필요
3. **log2+정규화**: 스케일 압축 + 샘플 분포 통일 (어레이는 RMA가 이미 함)
4. **ComBat**: 데이터셋 간 배치효과 제거 후 합침